# DuckDB mot Delta-tabeller på MinIO

Denne notebooken viser hvordan DuckDB kan brukes til å spørre Delta-tabeller
direkte fra MinIO — uten å starte en full Spark-kontekst.

**Stack:**
- DuckDB med `httpfs`-extension (S3-tilkobling) og `delta`-extension (Delta Lake-lesing)
- MinIO som S3-kompatibelt objektlager
- Gold-tabellen `department_stats` bygget av Airflow-pipelinen

**Sammenlignet med Spark:**
DuckDB er et in-process analytisk databasesystem — alt kjører i Python-prosessen.
Det er raskt å starte og effektivt for interaktiv analyse, men skalerer ikke til
distribuert prosessering slik Spark gjør.

## 1. Oppsett — DuckDB-tilkobling og MinIO-konfigurasjon

In [1]:
import duckdb

con = duckdb.connect()

# Last inn extensions for S3 og Delta Lake
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL delta;  LOAD delta;")

# Konfigurer MinIO som S3-endepunkt
con.execute("""
    CREATE OR REPLACE SECRET minio (
        TYPE        S3,
        KEY_ID      'admin',
        SECRET      'changeme',
        ENDPOINT    'minio:9000',
        URL_STYLE   'path',
        USE_SSL     false
    );
""")

GOLD_PATH = 's3://gold/department_stats'
print("DuckDB klar.", duckdb.__version__)

DuckDB klar. 1.5.1


## 2. Verifiser tabellstruktur

In [2]:
# Se alle kolonner og typer
con.execute(f"DESCRIBE SELECT * FROM delta_scan('{GOLD_PATH}')").df()

,column_name,column_type,null,key,default,extra
0,department,VARCHAR,YES,None,None,None
1,headcount,BIGINT,YES,None,None,None
2,avg_salary,DOUBLE,YES,None,None,None
3,min_salary,INTEGER,YES,None,None,None
4,max_salary,INTEGER,YES,None,None,None
5,total_payroll,BIGINT,YES,None,None,None
6,_gold_updated_at,TIMESTAMP WITH TIME ZONE,YES,None,None,None


## 3. Analytiske queries

In [4]:
# Query 1: Alle avdelinger sortert etter gjennomsnittlig lønn
con.execute(f"""
    SELECT
        department,
        headcount,
        avg_salary,
        ROUND(avg_salary / SUM(avg_salary) OVER () * 100, 1) AS pct_of_total_avg
    FROM delta_scan('{GOLD_PATH}')
    ORDER BY avg_salary DESC
""").df()

,department,headcount,avg_salary,pct_of_total_avg
0,engineering,4,102500.0,44.2
1,marketing,2,70000.0,30.2
2,sales,2,59500.0,25.6


In [5]:
# Query 2: Avdelinger med lønnsrom (maks - min) og antall ansatte
con.execute(f"""
    SELECT
        department,
        headcount,
        min_salary,
        max_salary,
        (max_salary - min_salary) AS salary_range
    FROM delta_scan('{GOLD_PATH}')
    ORDER BY salary_range DESC
""").df()

,department,headcount,min_salary,max_salary,salary_range
0,engineering,4,95000,112000,17000
1,marketing,2,68000,72000,4000
2,sales,2,58000,61000,3000


In [7]:
# Query 3: Totalt lønnskostnader per avdeling (approx: avg_salary * antall)
con.execute(f"""
    SELECT
        department,
        headcount,
        ROUND(avg_salary * headcount)           AS estimated_total_salary,
        ROUND(avg_salary * headcount /
              SUM(avg_salary * headcount) OVER () * 100, 1) AS pct_of_payroll
    FROM delta_scan('{GOLD_PATH}')
    ORDER BY estimated_total_salary DESC
""").df()

,department,headcount,estimated_total_salary,pct_of_payroll
0,engineering,4,410000.0,61.3
1,marketing,2,140000.0,20.9
2,sales,2,119000.0,17.8


In [9]:
# Query 4: Rangering av avdelinger etter antall ansatte (RANK over window)
con.execute(f"""
    SELECT
        department,
        headcount,
        RANK() OVER (ORDER BY headcount DESC)  AS headcount_rank,
        RANK() OVER (ORDER BY avg_salary DESC)      AS salary_rank
    FROM delta_scan('{GOLD_PATH}')
    ORDER BY headcount_rank
""").df()

,department,headcount,headcount_rank,salary_rank
0,engineering,4,1,1
1,marketing,2,2,2
2,sales,2,2,3


## 4. Ytelsessammenligning: DuckDB vs Spark

Vi måler tid for den samme aggregeringen i begge motorer.

In [10]:
import time

QUERY = f"""
    SELECT department, AVG(avg_salary) AS mean_salary
    FROM delta_scan('{GOLD_PATH}')
    GROUP BY department
    ORDER BY mean_salary DESC
"""

# -- DuckDB --
t0 = time.perf_counter()
duck_result = con.execute(QUERY).df()
duck_ms = (time.perf_counter() - t0) * 1000

print(f"DuckDB:  {duck_ms:.0f} ms  ({len(duck_result)} rader)")

DuckDB:  23 ms  (3 rader)


In [11]:
from pyspark.sql import SparkSession
import time

spark = (
    SparkSession.builder
    .appName("duckdb_vs_spark")
    .config("spark.sql.extensions",            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# -- Spark --
t0 = time.perf_counter()
spark_result = (
    spark.read.format("delta").load("s3a://gold/department_stats")
    .groupBy("department")
    .avg("avg_salary")
    .orderBy("avg(avg_salary)", ascending=False)
)
spark_result.collect()   # trigger faktisk kjøring
spark_ms = (time.perf_counter() - t0) * 1000

print(f"Spark:   {spark_ms:.0f} ms")
print(f"\nDuckDB er {spark_ms / duck_ms:.1f}x {'raskere' if duck_ms < spark_ms else 'tregere'} enn Spark for denne spørringen")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 07:24:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Spark:   5702 ms

DuckDB er 249.0x raskere enn Spark for denne spørringen


## 5. Oppsummering

| Motor  | Startoverhead | Skalering          | Brukstilfelle                           |
|--------|--------------|--------------------|-----------------------------------------|
| DuckDB | Lav (ms)     | Én maskin          | Interaktiv analyse, < ~100 GB           |
| Spark  | Høy (sekunder) | Distribuert klynge | Store datasett, komplekse transformasjoner |

**Konklusjon:** DuckDB er raskere for interaktiv spørring mot small-to-medium datasett
fordi det ikke trenger å starte JVM, fordele oppgaver til workers eller serialisere
data over nettverket. Spark vinner når dataene overskrider én maskins minne/CPU
eller når transformasjonene er del av en større, orkestrert pipeline.